# ⚡ Advanced Machine Learning: Gradient Boosting, Pipelines & Beyond

---

**Author:** Data Science Tutorial Series  
**Prerequisites:** Basic scikit-learn, pandas, numpy  
**Estimated Time:** 2–3 hours

> This notebook covers production-grade ML techniques that separate beginner
> projects from professional work: gradient boosting frameworks, advanced feature
> engineering, handling imbalanced data, sklearn pipelines, model stacking, and
> model interpretation.

All data in this notebook is **synthetically generated** for reproducibility.

## 📚 What You'll Learn

| Part | Topic | Key Skills |
|------|-------|-----------|
| **1** | Gradient Boosting Mastery | XGBoost, LightGBM, hyperparameter tuning |
| **2** | Advanced Feature Engineering | Polynomial features, target encoding, feature selection |
| **3** | Handling Imbalanced Data | Class weights, SMOTE, precision-recall analysis |
| **4** | Sklearn Pipelines & Stacking | ColumnTransformer, Pipeline, StackingClassifier |
| **5** | Model Interpretation | Feature importance methods, partial dependence |

Let's begin! 🚀

## 🔧 Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, cross_val_score, RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score,
    f1_score
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import (
    StandardScaler, PolynomialFeatures, OneHotEncoder, LabelEncoder
)
from sklearn.feature_selection import (
    mutual_info_classif, RFE, SelectKBest
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from scipy.stats import uniform, randint

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 5)

SEED = 42
np.random.seed(SEED)
print("✅ All imports successful!")

---
# Part 1 — Gradient Boosting Mastery 🌲

## 1.1 What is Gradient Boosting?

Gradient boosting builds an **ensemble of weak learners** (typically decision
trees) sequentially. Each new tree corrects the errors of the previous ensemble
by fitting to the **negative gradient** of the loss function.

Key advantages over Random Forests:
- Often achieves **higher accuracy** on tabular data
- Handles **mixed feature types** naturally
- Built-in **regularization** (learning rate, tree constraints)

We'll compare three major implementations:
| Framework | Strengths |
|-----------|-----------|
| **XGBoost** | Mature, excellent regularization, histogram-based option |
| **LightGBM** | Fastest training, leaf-wise growth, native categorical support |
| **CatBoost** | Best categorical handling, ordered boosting (not installed here) |

## 1.2 XGBoost — eXtreme Gradient Boosting

XGBoost introduced several innovations:
- **Regularized objective** (L1 & L2 on leaf weights)
- **Weighted quantile sketch** for approximate split finding
- **Sparsity-aware** split finding for missing values
- **Column subsampling** (borrowed from Random Forests)

Let's generate a synthetic classification dataset and train our first model.

In [ ]:
# Generate synthetic classification data
X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=12,
    n_redundant=4, n_clusters_per_class=2,
    flip_y=0.05, random_state=SEED
)

feature_names = [f"feat_{i}" for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Class distribution: {np.bincount(y_train)}")

In [ ]:
# Train XGBoost classifier
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    eval_metric="logloss",
    verbosity=0,
)

start = time.time()
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(f"XGBoost Training Time: {xgb_time:.3f}s")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_xgb):.4f}")
print()
print(classification_report(y_test, y_pred_xgb))

### Feature Importance (XGBoost)

XGBoost provides built-in feature importance based on **gain** — the average
improvement in the loss function when a feature is used for a split.

In [ ]:
# Feature importance plot — XGBoost
importances = xgb_model.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color="teal")
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

### XGBoost vs Random Forest

Let's compare XGBoost against a tuned Random Forest on the same data.

In [ ]:
# Random Forest comparison
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_leaf=4, random_state=SEED, n_jobs=-1
)
start = time.time()
rf_model.fit(X_train, y_train)
rf_time = time.time() - start
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "Model": ["XGBoost", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_rf)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_xgb),
        roc_auc_score(y_test, y_prob_rf)
    ],
    "Train Time (s)": [xgb_time, rf_time]
})
print(comparison.to_string(index=False))

## 1.3 LightGBM — Light Gradient Boosting Machine

LightGBM from Microsoft differs from XGBoost in key ways:
- **Leaf-wise** tree growth (vs. level-wise) → deeper, more accurate trees
- **Gradient-based One-Side Sampling (GOSS)** → faster training by focusing on
  samples with large gradients
- **Exclusive Feature Bundling (EFB)** → bundles sparse features to reduce
  dimensionality
- **Native categorical feature** support (no one-hot encoding needed)

In practice, LightGBM is often **2–10× faster** than XGBoost with comparable
or better accuracy.

In [ ]:
# Train LightGBM
lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    verbosity=-1,
)

start = time.time()
lgbm_model.fit(X_train, y_train)
lgbm_time = time.time() - start

y_pred_lgbm = lgbm_model.predict(X_test)
y_prob_lgbm = lgbm_model.predict_proba(X_test)[:, 1]

print(f"LightGBM Training Time: {lgbm_time:.3f}s")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lgbm):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lgbm):.4f}")
print()

# Three-way comparison
comparison = pd.DataFrame({
    "Model": ["XGBoost", "LightGBM", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_lgbm),
        accuracy_score(y_test, y_pred_rf),
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_xgb),
        roc_auc_score(y_test, y_prob_lgbm),
        roc_auc_score(y_test, y_prob_rf),
    ],
    "Train Time (s)": [xgb_time, lgbm_time, rf_time],
})
print(comparison.to_string(index=False))

## 1.4 CatBoost — Categorical Boosting (Overview)

**CatBoost** (by Yandex) is the third major gradient boosting library. While not
installed in this environment, it's important to know when to use it:

### Key Features
- **Ordered Target Statistics** for categorical encoding — avoids target leakage
  that plagues naive target encoding
- **Ordered Boosting** — a modification of gradient boosting that reduces
  prediction shift (a form of overfitting)
- **Native GPU training** with excellent multi-GPU scaling
- **Built-in text and embedding** feature support

### When to Choose CatBoost
| Scenario | Use CatBoost? |
|----------|--------------|
| Many high-cardinality categorical features | ✅ Best choice |
| Need minimal preprocessing | ✅ Handles cats & missing values |
| Large-scale GPU training | ✅ Excellent GPU support |
| Need fastest training on CPU | ❌ LightGBM is faster |
| Kaggle competition (tabular) | ✅ Often top 3 with XGB/LGBM |

```python
# Example usage (if installed):
from catboost import CatBoostClassifier
model = CatBoostClassifier(
    iterations=500, depth=6, learning_rate=0.1,
    cat_features=['city', 'category'],  # just pass column names!
    verbose=100
)
model.fit(X_train, y_train)
```

## 1.5 Hyperparameter Tuning with RandomizedSearchCV

The most impactful XGBoost hyperparameters to tune:

| Parameter | Effect | Typical Range |
|-----------|--------|--------------|
| `max_depth` | Tree complexity | 3–10 |
| `learning_rate` | Step size shrinkage | 0.01–0.3 |
| `n_estimators` | Number of boosting rounds | 100–1000 |
| `subsample` | Row sampling ratio | 0.5–1.0 |
| `colsample_bytree` | Column sampling ratio | 0.5–1.0 |
| `reg_alpha` | L1 regularization | 0–10 |
| `reg_lambda` | L2 regularization | 0–10 |
| `min_child_weight` | Min sum of instance weight in a child | 1–10 |

We'll use `RandomizedSearchCV` which samples from parameter distributions rather
than exhaustively searching a grid — much more efficient for large search spaces.

In [ ]:
# Hyperparameter tuning with RandomizedSearchCV
param_distributions = {
    "max_depth": randint(3, 10),
    "learning_rate": uniform(0.01, 0.29),
    "n_estimators": randint(100, 500),
    "subsample": uniform(0.5, 0.5),
    "colsample_bytree": uniform(0.5, 0.5),
    "reg_alpha": uniform(0, 5),
    "reg_lambda": uniform(0, 5),
    "min_child_weight": randint(1, 10),
}

rs_cv = RandomizedSearchCV(
    XGBClassifier(eval_metric="logloss", verbosity=0, random_state=SEED),
    param_distributions=param_distributions,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring="roc_auc",
    random_state=SEED,
    n_jobs=-1,
    verbose=0,
)

rs_cv.fit(X_train, y_train)

print("Best ROC-AUC (CV):", f"{rs_cv.best_score_:.4f}")
print("\nBest Parameters:")
for param, value in sorted(rs_cv.best_params_.items()):
    print(f"  {param}: {value:.4f}" if isinstance(value, float) else f"  {param}: {value}")

In [ ]:
# Visualize parameter importance from RandomizedSearchCV results
results_df = pd.DataFrame(rs_cv.cv_results_)

# Extract the top parameters by correlation with mean_test_score
param_cols = [c for c in results_df.columns if c.startswith("param_")]
correlations = {}
for col in param_cols:
    vals = pd.to_numeric(results_df[col], errors="coerce")
    if vals.notna().sum() > 5:
        correlations[col.replace("param_", "")] = abs(vals.corr(results_df["mean_test_score"]))

corr_series = pd.Series(correlations).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
corr_series.plot.barh(ax=ax, color="coral")
ax.set_xlabel("Absolute Correlation with CV Score")
ax.set_title("Hyperparameter Importance (Correlation with Performance)")
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate tuned model on test set
y_pred_tuned = rs_cv.best_estimator_.predict(X_test)
y_prob_tuned = rs_cv.best_estimator_.predict_proba(X_test)[:, 1]

print("Tuned XGBoost Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, y_prob_tuned):.4f}")
print(f"\nDefault XGBoost Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"  ROC-AUC:  {roc_auc_score(y_test, y_prob_xgb):.4f}")

---
# Part 2 — Advanced Feature Engineering 🔩

> *"Feature engineering is the art of extracting more signal from your data.
> A good feature can be worth more than a better algorithm."*

In this section we'll explore techniques that go beyond basic scaling and encoding.

## 2.1 Polynomial Features

`PolynomialFeatures` creates interaction terms and powers of existing features.
For degree 2 with features \(x_1, x_2\), it generates: \(1, x_1, x_2, x_1^2, x_1 x_2, x_2^2\).

This is especially helpful for **linear models** that can't capture non-linear
relationships natively.

In [ ]:
# Demonstrate polynomial features on a simple non-linear problem
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Create data with a non-linear decision boundary
np.random.seed(SEED)
n = 800
r = np.random.randn(n)
theta = np.random.uniform(0, 2 * np.pi, n)
X_circle = np.column_stack([r * np.cos(theta), r * np.sin(theta)])
y_circle = (r > 0.8).astype(int)  # circular boundary

X_tr, X_te, y_tr, y_te = train_test_split(
    X_circle, y_circle, test_size=0.25, random_state=SEED
)

# Linear model WITHOUT polynomial features
lr_plain = LogisticRegression(random_state=SEED)
lr_plain.fit(X_tr, y_tr)
acc_plain = accuracy_score(y_te, lr_plain.predict(X_te))

# Linear model WITH polynomial features (degree=3)
poly_pipe = make_pipeline(PolynomialFeatures(degree=3, include_bias=False), LogisticRegression(random_state=SEED))
poly_pipe.fit(X_tr, y_tr)
acc_poly = accuracy_score(y_te, poly_pipe.predict(X_te))

print(f"Logistic Regression (linear):      {acc_plain:.4f}")
print(f"Logistic Regression (poly deg=3):  {acc_poly:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model, title in zip(axes, [lr_plain, poly_pipe],
                              ["Linear LR", "Polynomial LR (deg=3)"]):
    xx, yy = np.meshgrid(
        np.linspace(X_circle[:, 0].min()-0.5, X_circle[:, 0].max()+0.5, 200),
        np.linspace(X_circle[:, 1].min()-0.5, X_circle[:, 1].max()+0.5, 200),
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_circle[:, 0], X_circle[:, 1], c=y_circle, cmap="coolwarm",
               edgecolors="k", s=15, alpha=0.6)
    ax.set_title(f"{title} — Acc: {accuracy_score(y_circle, model.predict(X_circle)):.3f}")
plt.tight_layout()
plt.show()

## 2.2 Target Encoding (Mean Encoding)

Target encoding replaces a categorical value with the **mean of the target**
for that category. It's powerful for high-cardinality features but carries a risk
of **target leakage**.

### The Concept
For a category `c` in feature `X`:
$$\text{TargetEncode}(c) = \frac{\sum_{i: X_i=c} y_i}{\sum_{i: X_i=c} 1}$$

### Avoiding Leakage
- Use **leave-one-out** encoding (exclude current row)
- Add **smoothing** (blend with global mean)
- Apply encoding using **cross-validation folds** (encode fold k using data from other folds)

In [ ]:
# Manual target encoding with smoothing
np.random.seed(SEED)
n_samples = 2000
cities = np.random.choice(["NYC", "LA", "Chicago", "Houston", "Phoenix",
                            "Philly", "Dallas", "Austin", "Denver", "Seattle"],
                           size=n_samples)
income = np.random.normal(50000, 15000, n_samples)
y_purchase = ((income > 55000).astype(int) +
              np.isin(cities, ["NYC", "LA", "Seattle"]).astype(int))
y_purchase = (y_purchase >= 1).astype(int)

df_te = pd.DataFrame({"city": cities, "income": income, "purchased": y_purchase})

def target_encode(df, col, target, smoothing=10):
    """Target encoding with global-mean smoothing."""
    global_mean = df[target].mean()
    agg = df.groupby(col)[target].agg(["mean", "count"])
    # Smoothed encoding: blend category mean with global mean
    smooth = (agg["count"] * agg["mean"] + smoothing * global_mean) / (agg["count"] + smoothing)
    return df[col].map(smooth)

df_te["city_encoded"] = target_encode(df_te, "city", "purchased", smoothing=10)

# Show the encoding
encoding_table = (df_te.groupby("city")
                  .agg(count=("purchased", "size"),
                       purchase_rate=("purchased", "mean"),
                       encoded_value=("city_encoded", "first"))
                  .sort_values("purchase_rate", ascending=False))
print(encoding_table.to_string())

## 2.3 Feature Interactions

Sometimes the **combination** of two features is more predictive than either
alone. For example, `height × weight` might predict health risk better than
either feature individually.

We can:
1. **Manually create** domain-specific interactions
2. **Automatically generate** them with `PolynomialFeatures(interaction_only=True)`
3. **Select the best** interactions using mutual information or model importance

In [ ]:
# Create interaction features and identify the best ones
X_interact, y_interact = make_classification(
    n_samples=3000, n_features=8, n_informative=5,
    n_redundant=1, random_state=SEED
)
feat_names_8 = [f"f{i}" for i in range(8)]
df_interact = pd.DataFrame(X_interact, columns=feat_names_8)

# Generate interaction terms (no powers, only cross-products)
poly_interact = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly_interact.fit_transform(X_interact)
poly_names = poly_interact.get_feature_names_out(feat_names_8)

print(f"Original features:    {X_interact.shape[1]}")
print(f"With interactions:    {X_poly.shape[1]}")

# Use mutual information to find the top interaction features
mi_scores = mutual_info_classif(X_poly, y_interact, random_state=SEED)
mi_df = pd.DataFrame({"feature": poly_names, "MI_score": mi_scores})
mi_df = mi_df.sort_values("MI_score", ascending=False)

print("\nTop 15 features by Mutual Information:")
print(mi_df.head(15).to_string(index=False))

## 2.4 Feature Selection

Too many features leads to overfitting, slower training, and harder
interpretability. Two powerful selection methods:

### Mutual Information (Filter Method)
Measures the **statistical dependency** between each feature and the target.
Works for both linear and non-linear relationships.

### Recursive Feature Elimination (Wrapper Method)
Trains a model, removes the least important feature, repeats.
Computationally expensive but finds feature **subsets** that work well together.

In [ ]:
# Feature selection: Mutual Information + RFE
from sklearn.feature_selection import mutual_info_classif, RFE

# 1. Mutual Information scores
mi = mutual_info_classif(X_train, y_train, random_state=SEED)
mi_ranking = pd.Series(mi, index=feature_names).sort_values(ascending=False)

# 2. RFE with Random Forest
rfe = RFE(
    estimator=RandomForestClassifier(n_estimators=50, random_state=SEED),
    n_features_to_select=10,
    step=2,
)
rfe.fit(X_train, y_train)
rfe_selected = [f for f, s in zip(feature_names, rfe.support_) if s]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# MI plot
mi_ranking.plot.barh(ax=axes[0], color="steelblue")
axes[0].set_xlabel("Mutual Information Score")
axes[0].set_title("Mutual Information Feature Ranking")
axes[0].invert_yaxis()

# RFE ranking
rfe_ranks = pd.Series(rfe.ranking_, index=feature_names).sort_values()
colors = ["green" if r == 1 else "lightgray" for r in rfe_ranks.values]
rfe_ranks.plot.barh(ax=axes[1], color=colors)
axes[1].set_xlabel("RFE Ranking (1 = selected)")
axes[1].set_title("RFE Feature Ranking")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print(f"\nRFE selected features ({len(rfe_selected)}): {rfe_selected}")

---
# Part 3 — Handling Imbalanced Data ⚖️

## The Problem

In many real-world tasks (fraud detection, medical diagnosis, rare-event
prediction), one class vastly outnumbers the other. A model that predicts
the majority class 100% of the time achieves **high accuracy but is useless**.

| Metric | What it tells you |
|--------|------------------|
| **Accuracy** | Misleading when classes are imbalanced |
| **Precision** | Of predicted positives, how many are truly positive? |
| **Recall** | Of actual positives, how many did we catch? |
| **F1 Score** | Harmonic mean of precision and recall |
| **PR-AUC** | Area under Precision-Recall curve — best single metric |

We'll explore three strategies:
1. **Baseline** (do nothing — see the problem)
2. **Class weights** (tell the model to pay more attention to minority)
3. **SMOTE** (synthesize new minority samples)

In [ ]:
# Create an imbalanced dataset (95% / 5%)
X_imb, y_imb = make_classification(
    n_samples=10000, n_features=20, n_informative=10,
    n_redundant=5, n_classes=2,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    flip_y=0.02, random_state=SEED
)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.2, stratify=y_imb, random_state=SEED
)

print("Training set class distribution:")
unique, counts = np.unique(y_train_imb, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt} ({cnt/len(y_train_imb)*100:.1f}%)")

## 3.1 Baseline — Ignoring the Imbalance

Let's train a default XGBoost model and see what happens.

In [ ]:
# Baseline model — no imbalance handling
xgb_baseline = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=SEED, eval_metric="logloss", verbosity=0
)
xgb_baseline.fit(X_train_imb, y_train_imb)
y_pred_base = xgb_baseline.predict(X_test_imb)

print("=== BASELINE (No Imbalance Handling) ===")
print(classification_report(y_test_imb, y_pred_base, digits=4))
print(f"Note: High accuracy ({accuracy_score(y_test_imb, y_pred_base):.4f}) "
      f"but check Class 1 recall ↑")

## 3.2 Class Weights

XGBoost's `scale_pos_weight` parameter controls the balance of positive and
negative weights. Setting it to `count(negative) / count(positive)` tells the
model to penalize misclassifying the minority class more heavily.

In [ ]:
# Class weights approach
neg_count = np.sum(y_train_imb == 0)
pos_count = np.sum(y_train_imb == 1)
scale_ratio = neg_count / pos_count

xgb_weighted = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_ratio,
    random_state=SEED, eval_metric="logloss", verbosity=0
)
xgb_weighted.fit(X_train_imb, y_train_imb)
y_pred_weighted = xgb_weighted.predict(X_test_imb)

print(f"=== CLASS WEIGHTS (scale_pos_weight={scale_ratio:.1f}) ===")
print(classification_report(y_test_imb, y_pred_weighted, digits=4))

## 3.3 SMOTE — Synthetic Minority Over-sampling Technique

SMOTE creates **synthetic** minority samples by interpolating between existing
minority instances and their nearest neighbors in feature space.

> ⚠️ **Important:** SMOTE must only be applied to the **training set**, never
> to the test set. Otherwise you get data leakage.

In [ ]:
# SMOTE approach
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imb, y_train_imb)

print("After SMOTE:")
unique, counts = np.unique(y_train_smote, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Class {cls}: {cnt}")

xgb_smote = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=SEED, eval_metric="logloss", verbosity=0
)
xgb_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = xgb_smote.predict(X_test_imb)

print(f"\n=== SMOTE ===")
print(classification_report(y_test_imb, y_pred_smote, digits=4))

## 3.4 Precision-Recall Curves — Comparing All Approaches

The Precision-Recall (PR) curve is the **gold standard** for evaluating
imbalanced classification. The area under the PR curve (PR-AUC) summarizes
performance across all thresholds.

In [ ]:
# Precision-Recall curves for all three approaches
fig, ax = plt.subplots(figsize=(9, 6))

models = {
    "Baseline": xgb_baseline,
    "Class Weights": xgb_weighted,
    "SMOTE": xgb_smote,
}
colors = ["gray", "steelblue", "coral"]

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test_imb)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test_imb, y_prob)
    ap = average_precision_score(y_test_imb, y_prob)
    ax.plot(rec, prec, label=f"{name} (AP={ap:.3f})", color=color, linewidth=2)

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves — Imbalanced Classification", fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

---
# Part 4 — Sklearn Pipelines & Stacking 🔗

## 4.1 Why Pipelines?

Without pipelines, you risk:
- **Data leakage** — fitting a scaler on the full dataset before splitting
- **Messy code** — separate transform/fit/predict steps that fall out of sync
- **Reproducibility issues** — forgetting a preprocessing step in production

A `Pipeline` chains preprocessing and modeling into a **single estimator** that
can be cross-validated, grid-searched, and serialized.

In [ ]:
# Create a mixed-type dataset (numeric + categorical) to showcase pipelines
np.random.seed(SEED)
n = 3000

df_pipe = pd.DataFrame({
    "age": np.random.randint(18, 80, n).astype(float),
    "income": np.random.lognormal(10.5, 0.8, n),
    "credit_score": np.random.normal(650, 80, n),
    "years_employed": np.random.exponential(5, n),
    "education": np.random.choice(["high_school", "bachelors", "masters", "phd"], n),
    "region": np.random.choice(["north", "south", "east", "west"], n),
})

# Inject some missing values
for col in ["age", "income", "credit_score"]:
    mask = np.random.random(n) < 0.05
    df_pipe.loc[mask, col] = np.nan

# Create target: probability of loan default
logit = (
    -3
    + 0.02 * df_pipe["age"].fillna(40)
    - 0.00002 * df_pipe["income"].fillna(40000)
    - 0.005 * df_pipe["credit_score"].fillna(650)
    + 0.05 * df_pipe["years_employed"].fillna(3)
    + 0.5 * (df_pipe["education"] == "high_school").astype(float)
)
prob = 1 / (1 + np.exp(-logit))
df_pipe["default"] = np.random.binomial(1, prob)

print(f"Dataset shape: {df_pipe.shape}")
print(f"Missing values:\n{df_pipe.isnull().sum()}")
print(f"\nTarget distribution:\n{df_pipe['default'].value_counts()}")
df_pipe.head()

### Building the Pipeline

We use `ColumnTransformer` to apply different preprocessing to numeric vs.
categorical columns, then chain it with a classifier.

In [ ]:
# Define column groups
numeric_features = ["age", "income", "credit_score", "years_employed"]
categorical_features = ["education", "region"]

# Numeric pipeline: impute missing → scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Categorical pipeline: impute missing → one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Full pipeline: preprocess → model
full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        random_state=SEED, eval_metric="logloss", verbosity=0
    ))
])

# Split and train
X_pipe = df_pipe.drop("default", axis=1)
y_pipe = df_pipe["default"]
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_pipe, y_pipe, test_size=0.2, stratify=y_pipe, random_state=SEED
)

full_pipeline.fit(X_tr_p, y_tr_p)
y_pred_pipe = full_pipeline.predict(X_te_p)
print(f"Pipeline Accuracy: {accuracy_score(y_te_p, y_pred_pipe):.4f}")
print(f"Pipeline F1:       {f1_score(y_te_p, y_pred_pipe):.4f}")
print("\nPipeline steps:")
for name, step in full_pipeline.steps:
    print(f"  {name}: {step.__class__.__name__}")

## 4.2 Stacking Classifier

Stacking combines multiple **diverse** models by training a **meta-learner** on
their predictions. The idea: different models capture different patterns, and the
meta-learner learns the optimal way to combine them.

```
Level 0:  LogisticRegression  →  pred_1 ─┐
          RandomForest        →  pred_2 ─┤→  Meta-learner  →  final prediction
          XGBoost             →  pred_3 ─┘
```

Scikit-learn's `StackingClassifier` handles the cross-validated training
automatically to avoid data leakage.

In [ ]:
# Stacking: combine LR + RF + XGBoost with a Logistic Regression meta-learner
base_estimators = [
    ("lr", Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED))
    ])),
    ("rf", Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=SEED))
    ])),
    ("xgb", Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            random_state=SEED, eval_metric="logloss", verbosity=0
        ))
    ])),
]

stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000, random_state=SEED),
    cv=5,
    n_jobs=-1,
)

stacking_clf.fit(X_tr_p, y_tr_p)
y_pred_stack = stacking_clf.predict(X_te_p)

print("=== Individual Model Results ===")
for name, pipe in base_estimators:
    pipe.fit(X_tr_p, y_tr_p)
    y_p = pipe.predict(X_te_p)
    print(f"  {name:4s}  Accuracy: {accuracy_score(y_te_p, y_p):.4f}  "
          f"F1: {f1_score(y_te_p, y_p):.4f}")

print(f"\n=== Stacking Ensemble ===")
print(f"  stack Accuracy: {accuracy_score(y_te_p, y_pred_stack):.4f}  "
      f"F1: {f1_score(y_te_p, y_pred_stack):.4f}")

## 4.3 Cross-Validation with Pipelines

The **correct** way to cross-validate: put all preprocessing inside the pipeline
so each fold is fit independently. This prevents data leakage from the scaler
or encoder seeing test-fold data.

```
❌ Wrong: scaler.fit(X) → split → model.fit(X_train) → model.predict(X_test)
✅ Right: split → pipeline.fit(X_train) → pipeline.predict(X_test)
```

In [ ]:
# Proper cross-validation with the full pipeline
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_scores = cross_val_score(
    full_pipeline, X_pipe, y_pipe, cv=cv,
    scoring="roc_auc", n_jobs=-1
)

print("5-Fold Cross-Validation (ROC-AUC):")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"  Mean:   {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
# Part 5 — Model Interpretation 🔍

Understanding **why** a model makes predictions is as important as the
predictions themselves. We'll explore three complementary approaches.

## 5.1 Feature Importance Methods

| Method | What it Measures | Pros | Cons |
|--------|-----------------|------|------|
| **MDI** (Mean Decrease in Impurity) | Avg impurity reduction per feature | Fast, built-in | Biased toward high-cardinality |
| **Permutation Importance** | Drop in score when feature is shuffled | Model-agnostic, reliable | Slow, correlated features share importance |
| **SHAP** | Shapley values from game theory | Theoretically grounded, local + global | Very slow for large datasets |

In [ ]:
# Compare MDI vs Permutation Importance
# Using the XGBoost model trained on the balanced dataset (Part 1)
# MDI importance
mdi_imp = pd.Series(xgb_model.feature_importances_, index=feature_names)

# Permutation importance
perm_result = permutation_importance(
    xgb_model, X_test, y_test,
    n_repeats=15, random_state=SEED, n_jobs=-1
)
perm_imp = pd.Series(perm_result.importances_mean, index=feature_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

mdi_sorted = mdi_imp.sort_values()
mdi_sorted.plot.barh(ax=axes[0], color="teal")
axes[0].set_title("MDI Feature Importance")
axes[0].set_xlabel("Importance (Gain)")

perm_sorted = perm_imp.sort_values()
perm_sorted.plot.barh(ax=axes[1], color="coral")
axes[1].set_title("Permutation Feature Importance")
axes[1].set_xlabel("Mean Accuracy Decrease")

plt.tight_layout()
plt.show()

# Agreement analysis
print("\nTop 5 features by each method:")
print(f"  MDI:         {list(mdi_imp.nlargest(5).index)}")
print(f"  Permutation: {list(perm_imp.nlargest(5).index)}")

### Manual SHAP-like Analysis

While SHAP requires a dedicated library, we can approximate the concept:
for each feature, measure how much the prediction changes when we replace it
with its marginal distribution (baseline value).

In [ ]:
# Simplified SHAP-like analysis: measure prediction shift per feature
baseline_probs = xgb_model.predict_proba(X_test)[:, 1]

shap_approx = {}
for feat in feature_names:
    X_permuted = X_test.copy()
    # Replace feature with shuffled values (breaks its relationship with target)
    X_permuted[feat] = np.random.permutation(X_permuted[feat].values)
    permuted_probs = xgb_model.predict_proba(X_permuted)[:, 1]
    # Mean absolute change in predicted probability
    shap_approx[feat] = np.mean(np.abs(baseline_probs - permuted_probs))

shap_series = pd.Series(shap_approx).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
shap_series.plot.barh(ax=ax, color="mediumpurple")
ax.set_xlabel("Mean |ΔP(y=1)| when feature is permuted")
ax.set_title("Approximate Feature Impact (SHAP-like)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5.2 Partial Dependence Plots

A Partial Dependence Plot (PDP) shows the **marginal effect** of one or two
features on the predicted outcome, averaging over all other features.

$$\hat{f}_{PDP}(x_s) = \frac{1}{N} \sum_{i=1}^{N} \hat{f}(x_s, x_{c}^{(i)})$$

Where $x_s$ is the feature of interest and $x_c$ are the complement features.

In [ ]:
# Manual Partial Dependence Plots for top 2 features
top_features = list(mdi_imp.nlargest(2).index)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, feat in zip(axes, top_features):
    # Create a grid of values for this feature
    grid = np.linspace(X_test[feat].min(), X_test[feat].max(), 50)
    pdp_values = []

    for val in grid:
        X_temp = X_test.copy()
        X_temp[feat] = val
        avg_pred = xgb_model.predict_proba(X_temp)[:, 1].mean()
        pdp_values.append(avg_pred)

    ax.plot(grid, pdp_values, color="steelblue", linewidth=2)
    ax.fill_between(grid, pdp_values, alpha=0.15, color="steelblue")
    ax.set_xlabel(feat, fontsize=12)
    ax.set_ylabel("Average P(y=1)", fontsize=12)
    ax.set_title(f"Partial Dependence: {feat}", fontsize=13)

    # Add rug plot
    ax.scatter(X_test[feat].values[:100], [min(pdp_values)]*100,
               marker="|", color="black", alpha=0.3, s=50)

plt.tight_layout()
plt.show()

## 5.3 Algorithm Selection Guide

| Algorithm | Best For | Handles Missing? | Handles Cats? | Speed | Interpretability |
|-----------|----------|:-:|:-:|:-:|:-:|
| **Logistic Regression** | Linear problems, baseline | ❌ | ❌ (needs encoding) | ⚡⚡⚡ | ⭐⭐⭐ |
| **Random Forest** | General-purpose, robust | ❌ | ❌ (needs encoding) | ⚡⚡ | ⭐⭐ |
| **XGBoost** | Competitions, production | ✅ | ❌ (needs encoding) | ⚡⚡ | ⭐⭐ |
| **LightGBM** | Large datasets, speed | ✅ | ✅ (native) | ⚡⚡⚡ | ⭐⭐ |
| **CatBoost** | High-cardinality categoricals | ✅ | ✅ (best) | ⚡⚡ | ⭐⭐ |
| **SVM** | Small datasets, text/NLP | ❌ | ❌ | ⚡ | ⭐ |
| **Neural Networks** | Images, text, sequences | ❌ | ❌ | ⚡ (GPU) | ⭐ |

### Rules of Thumb
- **Start with LightGBM** for any tabular dataset → fast iteration
- **Use XGBoost** if you need maximum control over regularization
- **Use CatBoost** if you have many categorical features
- **Use stacking** when you need every last bit of performance
- **Use pipelines** always — no excuses for data leakage

---
## 🏋️ Exercise: Build a Complete Imbalanced Classification Pipeline

**Your task:** Using the synthetic dataset below, build a full pipeline that:

1. Preprocesses numeric and categorical features (handle missing values!)
2. Applies SMOTE to the training data
3. Uses a StackingClassifier with at least 2 base models
4. Evaluates using PR-AUC and F1 score
5. Compares results with and without SMOTE

**Hint:** Since `imblearn` provides a `Pipeline` that supports samplers,
you can use `from imblearn.pipeline import Pipeline as ImbPipeline` to include
SMOTE directly in your pipeline.

In [ ]:
# Exercise dataset — imbalanced with mixed feature types
np.random.seed(123)
n_ex = 5000

exercise_df = pd.DataFrame({
    "feature_A": np.random.normal(0, 1, n_ex),
    "feature_B": np.random.exponential(2, n_ex),
    "feature_C": np.random.uniform(-5, 5, n_ex),
    "feature_D": np.random.choice(["low", "medium", "high"], n_ex),
    "feature_E": np.random.choice(["typeA", "typeB", "typeC", "typeD"], n_ex),
    "feature_F": np.random.normal(10, 3, n_ex),
})

# Inject missing values
for col in ["feature_A", "feature_B", "feature_F"]:
    exercise_df.loc[np.random.random(n_ex) < 0.08, col] = np.nan

# Create imbalanced target (93/7 split)
logit_ex = (
    0.5 * exercise_df["feature_A"].fillna(0)
    - 0.3 * exercise_df["feature_C"]
    + 0.2 * (exercise_df["feature_D"] == "high").astype(float)
    - 2.5
)
prob_ex = 1 / (1 + np.exp(-logit_ex))
exercise_df["target"] = np.random.binomial(1, prob_ex)

print(f"Exercise dataset shape: {exercise_df.shape}")
print(f"\nTarget distribution:")
print(exercise_df["target"].value_counts())
print(f"\nMissing values:")
print(exercise_df.isnull().sum())
print("\n✏️ Your code goes below! Build the pipeline and evaluate it.")

In [ ]:
# YOUR SOLUTION HERE
# -------------------
# Suggested structure:
#
# from imblearn.pipeline import Pipeline as ImbPipeline
#
# numeric_features_ex = ["feature_A", "feature_B", "feature_C", "feature_F"]
# categorical_features_ex = ["feature_D", "feature_E"]
#
# preprocessor_ex = ColumnTransformer(...)
#
# pipeline_ex = ImbPipeline([
#     ("preprocessor", preprocessor_ex),
#     ("smote", SMOTE(random_state=42)),
#     ("classifier", StackingClassifier(...))
# ])
#
# # Split data
# # Fit pipeline
# # Evaluate with PR-AUC and F1
#
# Good luck! 🍀
pass

---
## 🚀 Next Steps

Congratulations on completing this advanced ML tutorial! Here's where to go next:

### Deep Learning
- **PyTorch** or **TensorFlow/Keras** for neural networks
- Start with tabular data (`nn.Module` with embeddings for categoricals)
- Move to CNNs (images) and RNNs/Transformers (text/sequences)

### Natural Language Processing
- **Hugging Face Transformers** — pre-trained BERT, GPT, etc.
- **Sentence-Transformers** for embeddings and semantic search
- Fine-tuning LLMs for classification, NER, summarization

### MLOps & Production
- **MLflow** — experiment tracking, model registry
- **Docker** — containerize your model serving
- **FastAPI** — build REST APIs for model inference
- **Great Expectations** — data validation in production

### Advanced Techniques
- **AutoML** — Auto-sklearn, FLAML, H2O
- **Bayesian Optimization** — Optuna for hyperparameter tuning
- **Explainability** — SHAP, LIME, Anchor explanations
- **Causal Inference** — DoWhy, EconML

---
*Happy modeling!* 🎉